In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# The pandas AI functions package requires OpenAI version 1.99.5 or later
%pip install -q --force-reinstall openai==1.99.5 2>/dev/null
# Required imports
import synapse.ml.aifunc as aifunc
import pandas as pd

StatementMeta(, cd3e3595-2157-4300-81a1-a224c4d1198c, 8, Finished, Available, Finished)

Note: you may need to restart the kernel to use updated packages.



In [4]:

import pandas as pd

# Load data into pandas DataFrame
df = pd.read_csv("/lakehouse/default/Files/codeserendipity/CallLogs/contoso_financial_support_logs_v3.csv")

# Remove the 'sentiment' column if it exists
if 'Sentiment' in df.columns:
    df = df.drop('Sentiment', axis=1)

display(df)

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, cd9656b7-a8e8-4f94-9b31-ac80f2a10db9)

In [5]:

# Remove duplicate rows from the DataFrame
df = df.drop_duplicates()

# Translate columns to English
df["Description_english"] = df["Description"].ai.translate("english")
df["Notes_english"] = df["Notes"].ai.translate("english")
df["CallTranscript_english"] = df["CallTranscript"].ai.translate("english")

display(df)

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 8279e425-921f-4f8e-b0b9-90f32a811d79)

100%|##########| 300/300 [00:07<00:00, 40.59it/s]


In [8]:
df["Sentiment"] = df["CallTranscript_english"].ai.analyze_sentiment()
display(df)

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7fb09cc0-e7c9-44c8-a322-4ce4f78236ab)

100%|##########| 300/300 [00:03<00:00, 90.41it/s]


In [9]:
df["Category"] = df["Description_english"].ai.classify("documents","scheduling","trading","reporting","bug")
display(df)

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 19, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f226aafc-1d42-483f-b83c-cb864c0e96c5)

100%|##########| 300/300 [00:03<00:00, 98.80it/s]


In [10]:
df["CallSummary"] = df["CallTranscript_english"].ai.summarize()
display(df)

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 20, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 040e6d0d-6516-4cab-94d2-eef65cb094d9)

100%|##########| 300/300 [00:05<00:00, 55.73it/s]


In [11]:

spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

# Check type and convert Pandas DataFrame to Spark DataFrame if needed
import pandas as pd

if isinstance(df, pd.DataFrame):
    spark_df = spark.createDataFrame(df)
else:
    spark_df = df

# Now, register as a temporary SQL view
spark_df.createOrReplaceTempView("my_temp_view")

# Example Spark SQL usage
result_df = spark.sql("SELECT * FROM my_temp_view")
display(result_df)

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 88f3a6c8-ecdd-4aa9-898d-fa2817831d88)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Did not pass numpy.dtype object
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.


In [14]:


import pandas as pd

# Only keep the 'IssueCategory' column and remove duplicates
if isinstance(df, pd.DataFrame):
    issue_df = df[["IssueCategory"]].drop_duplicates().copy()
    spark_df = spark.createDataFrame(issue_df)
else:
    spark_df = df.select("IssueCategory").dropDuplicates()

# Save the DataFrame as a Delta table named "Call_Issue_Category" in the default Lakehouse
spark_df.write.mode("overwrite").format("delta").saveAsTable("Call_Issue_Category")

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 25, Finished, Available, Finished)

In [15]:
import pandas as pd

# Only keep the 'IssueCategory' column to create the table
if isinstance(df, pd.DataFrame):
    issue_df = df[["DeskName"]].drop_duplicates().copy()
    spark_df = spark.createDataFrame(issue_df)
else:
    spark_df = df.select("DeskName").drop_duplicates()

# Save the DataFrame as a Delta table named "Call_Issue_Category" in the default Lakehouse
spark_df.write.mode("overwrite").format("delta").saveAsTable("Call_Desk_Name")

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 26, Finished, Available, Finished)

In [16]:

# Only keep the 'IssueCategory' column to create the table
if isinstance(df, pd.DataFrame):
    issue_df = df[["AgentName"]].drop_duplicates().copy()
    spark_df = spark.createDataFrame(issue_df)
else:
    spark_df = df.select("AgentName").drop_duplicates()

# Save the DataFrame as a Delta table named "Call_Issue_Category" in the default Lakehouse
spark_df.write.mode("overwrite").format("delta").saveAsTable("Call_Agent_Name")

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 27, Finished, Available, Finished)

In [17]:
# Save a DataFrame called df to the Lakehouse table "callLogs" using PySpark
result_df.write.mode("overwrite").format("delta").save("Tables/callLogs")

StatementMeta(, c5e5a381-bcf5-4fb1-abeb-08bdc73fe9dc, 28, Finished, Available, Finished)